# CP-VTON+ quick test: user images -> SCHP parsing -> generated inputs -> try-on

This notebook follows the original dataset notebook, but it does **not** load `CPDataset`. It starts from two user-provided images only:

- `photo_full.png`
- `jacket.png`

Human parsing is generated with the packaged FP32 SCHP LIP model in `models/schp_hf`, then CP-VTON+ masks are derived from LIP labels.


In [ ]:
from pathlib import Path
import warnings
import torch
import matplotlib.pyplot as plt

import importlib
import tryon_preprocess
import tryon_parse_hf
import tryon_infer

# Reload local modules so a running notebook kernel picks up code edits.
importlib.reload(tryon_parse_hf)
importlib.reload(tryon_preprocess)
importlib.reload(tryon_infer)

from tryon_infer import TryOnModel
from tryon_preprocess import prepare_tryon_inputs, tensor_to_pil
from tryon_parse_hf import LIP_LABELS, PARSE_CLOTH_LABELS, PHEAD_TOM_LABELS

# ── Paths ──────────────────────────────────────────────────────────────────
PERSON_PATH = Path("photo_full.png")   # change to your photo
# CLOTH_PATH  = Path("jacket.png")        # change to your garment
CLOTH_PATH  = Path("tshirt.webp")        # change to your garment
GMM_CKPT    = Path("../cp-vton-plus/checkpoints/GMM/gmm_final.pth")
TOM_CKPT    = Path("../cp-vton-plus/checkpoints/TOM/tom_final.pth")
OUT_DIR     = Path("backend/output")
OUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device     :", DEVICE)
print("person     :", PERSON_PATH.exists(), PERSON_PATH)
print("cloth      :", CLOTH_PATH.exists(),  CLOTH_PATH)
print("GMM ckpt   :", GMM_CKPT.exists(),   GMM_CKPT)
print("TOM ckpt   :", TOM_CKPT.exists(),   TOM_CKPT)


## Step 1 — Run preprocessing pipeline

In [ ]:
warnings.filterwarnings("ignore")

sample = prepare_tryon_inputs(PERSON_PATH, CLOTH_PATH, device=DEVICE)

print("These tensors are the tensors used by model.tryon() below.")
print("image            :", tuple(sample.image.shape),
      f"{sample.image.min().item():.3f}..{sample.image.max().item():.3f}")
print("agnostic         :", tuple(sample.agnostic.shape),
      f"{sample.agnostic.min().item():.3f}..{sample.agnostic.max().item():.3f}")
print("cloth            :", tuple(sample.cloth.shape),
      f"{sample.cloth.min().item():.3f}..{sample.cloth.max().item():.3f}")
print("cloth_mask       :", tuple(sample.cloth_mask.shape),
      f"{sample.cloth_mask.min().item():.3f}..{sample.cloth_mask.max().item():.3f}")
print("shape            :", tuple(sample.shape.shape),
      f"{sample.shape.min().item():.3f}..{sample.shape.max().item():.3f}")
print("head             :", tuple(sample.head.shape),
      f"{sample.head.min().item():.3f}..{sample.head.max().item():.3f}")
print("parse_cloth      :", tuple(sample.parse_cloth.shape),
      f"{sample.parse_cloth.min().item():.3f}..{sample.parse_cloth.max().item():.3f}")
print("parse_cloth_mask :", tuple(sample.parse_cloth_mask.shape),
      f"{sample.parse_cloth_mask.min().item():.3f}..{sample.parse_cloth_mask.max().item():.3f}")
print("pose_image       :", tuple(sample.pose_image.shape),
      f"{sample.pose_image.min().item():.3f}..{sample.pose_image.max().item():.3f}")
print("parse_labels     :", sample.parse_labels.shape, sample.parse_labels.dtype,
      int(sample.parse_labels.min()), int(sample.parse_labels.max()))

assert torch.equal(sample.agnostic[0:1], sample.shape)
assert torch.equal(sample.agnostic[1:4], sample.head)


## Step 2 — Visualise generated inputs

In [ ]:
import numpy as np

def to_img01(x):
    if hasattr(x, 'size') and not hasattr(x, 'shape'):  # PIL Image
        return np.array(x) / 255.0
    if isinstance(x, np.ndarray):
        arr = x
        if arr.ndim == 2:
            arr = np.repeat(arr[:, :, None], 3, axis=2)
        return arr.astype(np.float32) / (255.0 if arr.max() > 1 else 1.0)
    if x.dim() == 4:
        x = x[0]
    x = x.detach().float().cpu()
    x = (x.clamp(-1, 1) + 1) * 0.5 if x.min() < -0.1 else x.clamp(0, 1)
    if x.shape[0] == 1:
        x = x.repeat(3, 1, 1)
    return x.permute(1, 2, 0).numpy()

def show_grid(items, cols=4, figsize=(14, 9)):
    if hasattr(items, 'items'):
        items = list(items.items())
    rows = (len(items) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = np.array(axes).reshape(-1)
    for i, (title, val) in enumerate(items):
        axes[i].imshow(to_img01(val))
        axes[i].set_title(title, fontsize=9)
        axes[i].axis('off')
    for j in range(i + 1, len(axes)):
        axes[j].axis('off')
    plt.tight_layout()
    plt.show()

pose_vis = sample.pose_image
agnostic_vis = sample.agnostic.max(dim=0, keepdim=True).values

# Same stage as tryon_test_from_dataset: these are visualizations of the exact
# tensors used to call model.tryon(sample.agnostic, sample.cloth, sample.cloth_mask).
show_grid({
    "person (image)": sample.image,
    "head": sample.head,
    "shape": sample.shape,
    "pose_image": pose_vis,
    "cloth": sample.cloth,
    "cloth_mask": sample.cloth_mask,
    "parse_cloth": sample.parse_cloth,
    "parse_cloth_mask": sample.parse_cloth_mask,
    "agnostic (max-ch)": agnostic_vis,
}, cols=4, figsize=(14, 9))


In [ ]:
LIP_COLORS = np.array([
    [0, 0, 0], [128, 0, 0], [255, 0, 0], [0, 85, 0], [170, 0, 51],
    [255, 85, 0], [0, 0, 85], [0, 119, 221], [85, 85, 0], [0, 85, 85],
    [85, 51, 0], [52, 86, 128], [0, 128, 0], [0, 0, 255],
    [51, 170, 221], [0, 255, 255], [85, 255, 170], [170, 255, 85],
    [255, 255, 0], [255, 170, 0],
], dtype=np.uint8)

parse_labels = sample.parse_labels
parse_vis = LIP_COLORS[np.clip(parse_labels, 0, len(LIP_COLORS) - 1)]
phead_mask = np.isin(parse_labels, list(PHEAD_TOM_LABELS)).astype(np.uint8) * 255
parse_cloth_mask_labels = np.isin(parse_labels, list(PARSE_CLOTH_LABELS)).astype(np.uint8) * 255
person_mask_labels = (parse_labels > 0).astype(np.uint8) * 255

unique, counts = np.unique(parse_labels, return_counts=True)
print('SCHP labels for the exact VTON person image:')
for label, count in zip(unique, counts):
    print(f'{int(label):2d} {LIP_LABELS[int(label)]:14s} {int(count):6d}')

show_grid({
    "person tensor image": sample.image,
    "raw LIP labels": parse_vis,
    "person mask labels 1..19": person_mask_labels,
    "phead mask labels": phead_mask,
    "head tensor": sample.head,
    "parse_cloth labels": parse_cloth_mask_labels,
    "parse_cloth tensor": sample.parse_cloth,
    "shape tensor": sample.shape,
}, cols=4, figsize=(14, 8))


In [ ]:
model = TryOnModel(
    gmm_ckpt=str(GMM_CKPT),
    tom_ckpt=str(TOM_CKPT),
    device=DEVICE,
    use_fp16=False,
)

out = model.tryon(
    sample.agnostic.unsqueeze(0),
    sample.cloth.unsqueeze(0),
    sample.cloth_mask.unsqueeze(0),
)
print("output shape:", tuple(out.shape),
      "  min/max:", round(out.min().item(), 3), round(out.max().item(), 3))

result = tensor_to_pil(out)
result_path = OUT_DIR / "notebook_user_images_tryon.jpg"
result.save(result_path, quality=92)
print("saved:", result_path)

## Step 4 — Preview result

In [ ]:
show_grid([
    ("person (input)",  sample.person_preview),
    ("cloth (input)",   sample.cloth_preview),
    ("cloth mask",      sample.cloth_mask_preview),
    ("try-on output",   out),
], cols=4, figsize=(14, 5))